# Fine-Tuning Comparison

Train SFT+LoRA, SFT+QLoRA, DPO, Reward Model, and GRPO on the StackOverflow Q&A dataset.

### Imports & Seed

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from datasets import Dataset
from src.utils.seed import set_seed
from src.data.preprocess import (
    load_stackoverflow, split_qa,
    make_sft_dataframe, make_preference_pairs,
    make_reward_pairs, make_grpo_prompts,
)

set_seed(42)

### Load and Split Data

In [ ]:
df = load_stackoverflow('../data/raw/stacksample')
train_df, val_df, test_df = split_qa(df)
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

### SFT + LoRA

In [ ]:
from src.training.sft import train_sft

sft_train = Dataset.from_pandas(make_sft_dataframe(train_df), preserve_index=False)
sft_val = Dataset.from_pandas(make_sft_dataframe(val_df), preserve_index=False)

train_sft(
    'google/gemma-3-270m', sft_train, sft_val,
    '../outputs/sft-lora', use_lora=True, qlora=False,
)

### SFT + QLoRA

In [ ]:
train_sft(
    'google/gemma-3-270m', sft_train, sft_val,
    '../outputs/sft-qlora', use_lora=True, qlora=True,
)

### DPO

In [ ]:
from src.training.dpo import train_dpo

dpo_df = make_preference_pairs(train_df)
dpo_train = Dataset.from_pandas(dpo_df.sample(frac=0.8, random_state=42), preserve_index=False)
dpo_val = Dataset.from_pandas(dpo_df.drop(dpo_train.to_pandas().index), preserve_index=False)

train_dpo('google/gemma-3-270m', dpo_train, dpo_val, '../outputs/dpo')

### Reward Modeling

In [ ]:
from src.training.reward import train_reward

rew_df = make_reward_pairs(train_df)
rew_train = Dataset.from_pandas(rew_df.sample(frac=0.8, random_state=42), preserve_index=False)
rew_val = Dataset.from_pandas(rew_df.drop(rew_train.to_pandas().index), preserve_index=False)

train_reward('google/gemma-3-270m', rew_train, rew_val, '../outputs/reward')

### GRPO

In [ ]:
from src.training.grpo import train_grpo

grpo_df = make_grpo_prompts(train_df)
grpo_train = Dataset.from_pandas(grpo_df.sample(frac=0.8, random_state=42), preserve_index=False)
grpo_val = Dataset.from_pandas(grpo_df.drop(grpo_train.to_pandas().index), preserve_index=False)

train_grpo('google/gemma-3-270m', grpo_train, grpo_val, '../outputs/grpo')

### Comparison Table

In [ ]:
results = pd.DataFrame([
    {'method': 'SFT+LoRA', 'output': '../outputs/sft-lora'},
    {'method': 'SFT+QLoRA', 'output': '../outputs/sft-qlora'},
    {'method': 'DPO', 'output': '../outputs/dpo'},
    {'method': 'Reward', 'output': '../outputs/reward'},
    {'method': 'GRPO', 'output': '../outputs/grpo'},
])
results